# 07 — Previsão de Rebaixamento 2025

Aplicação do modelo final (Regressão Logística treinada em 2014–2024) para prever os clubes com maior risco de rebaixamento na temporada 2025.

## Carregamento do Modelo Final Salvo

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
import os

FEATURES = ['Plantel', 'Estrangeiros', 'Valor de Mercado Total']

modelo  = joblib.load(os.path.join('..','modelos','logistica.pkl'))
scaler  = joblib.load(os.path.join('..','modelos','scaler_logistica.pkl'))
print('Modelo carregado:', type(modelo).__name__)
print('Coeficientes:', dict(zip(FEATURES, modelo.coef_[0].round(4))))

## Dados dos Clubes para 2025

In [ ]:
df = pd.read_excel(os.path.join('..','dados','BASE_FINAL.xlsx'), sheet_name='CLUBES')
df.columns = df.columns.str.strip()
df_2025 = df[df['Temporada']==2025].copy()

if df_2025.empty:
    print('Aviso: sem dados para 2025. Usando 2024 como proxy.')
    df_2025 = df[df['Temporada']==2024].copy()

print(f'Clubes para previsao: {len(df_2025)}')
df_2025[['Clube'] + FEATURES]

## Geração das Probabilidades de Rebaixamento

In [ ]:
X_2025 = scaler.transform(df_2025[FEATURES])
probs  = modelo.predict_proba(X_2025)
idx_reb = list(modelo.classes_).index(0)

df_prev = df_2025[['Clube'] + FEATURES].copy()
df_prev['Prob_Rebaixamento(%)']  = (probs[:, idx_reb] * 100).round(2)
df_prev['Prob_Permanencia(%)']   = (probs[:, 1-idx_reb] * 100).round(2)
df_prev['Previsao']              = np.where(df_prev['Prob_Rebaixamento(%)'] >= 50, 'Rebaixado', 'Permanece')

df_prev = df_prev.sort_values('Prob_Rebaixamento(%)', ascending=False).reset_index(drop=True)
df_prev.index += 1

# Marca os 4 primeiros como rebaixados (independente do limiar)
df_prev.loc[df_prev.index <= 4, 'Previsao'] = 'Rebaixado'

print('Tabela de Previsao 2025:')
df_prev[['Clube','Prob_Rebaixamento(%)','Prob_Permanencia(%)','Previsao']]

## Gráfico de Barras Horizontal — Ranking de Risco

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))

cores_bar = ['#e74c3c' if p=='Rebaixado' else '#3498db' for p in df_prev['Previsao']]
bars = ax.barh(
    df_prev['Clube'][::-1],
    df_prev['Prob_Rebaixamento(%)'][::-1],
    color=cores_bar[::-1], edgecolor='white', height=0.7
)
ax.axvline(x=50, color='gray', linestyle='--', alpha=0.5, label='Limiar 50%')
ax.set_xlabel('Probabilidade de Rebaixamento (%)', fontsize=12)
ax.set_title(f'Previsao de Rebaixamento — Brasileirao Serie A 2025', fontsize=14, fontweight='bold')
ax.set_xlim(0, 105)
for bar, val in zip(bars, df_prev['Prob_Rebaixamento(%)'][::-1]):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)
ax.legend(handles=[
    mpatches.Patch(color='#e74c3c', label='Rebaixado (top 4 risco)'),
    mpatches.Patch(color='#3498db', label='Permanece na Serie A')
], loc='lower right')
plt.tight_layout()
os.makedirs(os.path.join('..','img'), exist_ok=True)
plt.savefig(os.path.join('..','img','previsao_2025.png'), dpi=150, bbox_inches='tight')
plt.show()

## Tabela Final de Resultados

In [ ]:
print('=' * 70)
print('  PREVISAO DE REBAIXAMENTO — BRASILEIRAO SERIE A 2025')
print('=' * 70)
print(f'  Modelo: Regressao Logistica | Treinado em: 2014-2024')
print('=' * 70)
for _, row in df_prev.iterrows():
    status = '*** REBAIXADO ***' if row['Previsao']=='Rebaixado' else 'Permanece'
    print(f"  {row.name:>2}. {row['Clube']:<25} {row['Prob_Rebaixamento(%)']:>6.1f}%  {status}")
print('=' * 70)

## Análise dos Resultados

**Clubes em zona de risco (4 primeiros):** são os que o modelo aponta como maior probabilidade de rebaixamento com base nas features de 2025 (tamanho do elenco, estrangeiros e valor de mercado).

**Limitações desta previsão:**
1. Dados de 2025 podem não refletir mudanças de elenco ao longo da temporada
2. O modelo não captura fatores externos (lesões, mudança técnica, calendário sobrecarregado)
3. A variável `Pontos` não está disponível antes do fim da temporada
4. Desempenho dentro de campo tem peso que as variáveis financeiras não capturam completamente

**Trabalhos futuros:**
- Incluir features de desempenho em campo (aproveitamento, saldo de gols)
- Aplicar modelos de séries temporais (LSTM, Prophet)
- Retreinar o modelo ao longo da temporada com dados parciais de pontuação